# 03a  -  Run `ai_extract` (per-input pricing path)

This notebook runs `ai_extract` on every transcript from notebook 02, and logs the predicted cost + run window for the reconciliation step.

**Why `ai_extract`:** `ai_extract`, `ai_parse_document`, and `ai_classify` are billed at a flat **DBU rate per 1K inputs** (per the [AI Functions pricing page](https://www.databricks.com/product/pricing/ai-functions)). They show up in [`system.billing.usage`](https://docs.databricks.com/en/admin/system-tables/billing.html) under `billing_origin_product = 'AI_FUNCTIONS'`. This is the canonical and predictable pricing model for production extraction workloads.

**What this notebook does:**
1. Reads job parameters and the transcripts table from notebook 02.
2. Computes the **predicted cost** using the per-input formula at low and medium complexity.
3. Calls `ai_extract` on every row, capturing the start/end timestamps as the reconciliation window.
4. Writes the structured output to `<catalog>.<schema>.extract_results`.
5. Appends a row to `<catalog>.<schema>.run_log` with the predictions and the window  -  used by notebook 04 for the actual-vs-predicted variance.

**Public references:**
- [AI Functions pricing](https://www.databricks.com/product/pricing/ai-functions)
- [`ai_extract` SQL function reference](https://docs.databricks.com/en/sql/language-manual/functions/ai_extract.html)
- [Cost-attribution tags via `custom_tags`](https://docs.databricks.com/en/admin/account-settings/usage-detail-tags.html)


## Read job parameters

Same parameter pattern as the earlier notebooks  -  nothing hardcoded.


In [ ]:
import datetime as dt, uuid

# Helper: timezone-aware UTC, then strip tz to keep parity with Spark's naive
# TimestampType columns. Replaces the deprecated dt.datetime.utcnow().
def utc_now():
    return dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)

dbutils.widgets.text("catalog",      "main")
dbutils.widgets.text("schema",       "ai_functions_cost_demo")
dbutils.widgets.text("sample_size",  "100")
dbutils.widgets.text("team_tag",     "ai-cost-demo")
dbutils.widgets.text("owner_email",  "you@example.com")

CATALOG     = dbutils.widgets.get("catalog")
SCHEMA      = dbutils.widgets.get("schema")
TEAM_TAG    = dbutils.widgets.get("team_tag")
OWNER_EMAIL = dbutils.widgets.get("owner_email")

TRANSCRIPTS_TABLE = f"{CATALOG}.{SCHEMA}.transcripts"
RESULTS_TABLE     = f"{CATALOG}.{SCHEMA}.extract_results"
RUN_LOG_TABLE     = f"{CATALOG}.{SCHEMA}.run_log"

RUN_ID = "ai_extract-" + utc_now().strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]
print(f"Run id:        {RUN_ID}")
print(f"Reads from:    {TRANSCRIPTS_TABLE}")
print(f"Writes to:     {RESULTS_TABLE}")
print(f"Logs to:       {RUN_LOG_TABLE}")


## Predict the cost  -  BEFORE running

The next cell computes the predicted cost using the [AI Functions pricing page](https://www.databricks.com/product/pricing/ai-functions) per-input formula:

```
cost_usd = (rows / 1000) x DBUs_per_1K_inputs x $/DBU
```

**Two complexity brackets** (per the pricing page):
- **Low**  -  30 DBUs / 1K inputs. Typical for short structured text like call transcripts.
- **Medium**  -  60 DBUs / 1K inputs. Typical for richer documents.

We compute both at **list price** and with the **50% promotion** that runs through 2026-06-30. The actual reconciliation in notebook 04 will tell us which bracket the platform classified our transcripts as.


In [ ]:
from pyspark.sql import functions as F

# Pricing constants - validated 2026-04-28 from
# https://www.databricks.com/product/pricing/ai-functions
USD_PER_DBU                  = 0.07
DBUS_PER_1K_LOW              = 30
DBUS_PER_1K_MEDIUM           = 60
PROMO_FACTOR                 = 0.5   # 50% off through 2026-06-30

stats = (
    spark.table(TRANSCRIPTS_TABLE)
        .select(
            F.count("*").alias("rows"),
            F.sum(F.length("transcript")).alias("total_chars"),
            F.avg(F.length("transcript")).alias("avg_chars"),
        ).collect()[0]
)
n_rows      = stats["rows"]
total_chars = stats["total_chars"]
avg_chars   = float(stats["avg_chars"])

def cost(n, dbus_per_1k, promo=False):
    c = (n / 1000.0) * dbus_per_1k * USD_PER_DBU
    return c * (PROMO_FACTOR if promo else 1.0)

pred_low_list      = cost(n_rows, DBUS_PER_1K_LOW)
pred_low_promo     = cost(n_rows, DBUS_PER_1K_LOW,    promo=True)
pred_medium_list   = cost(n_rows, DBUS_PER_1K_MEDIUM)
pred_medium_promo  = cost(n_rows, DBUS_PER_1K_MEDIUM, promo=True)

print(f"Rows:                  {n_rows:,}")
print(f"Avg transcript chars:  {avg_chars:,.0f}")
print()
print(f"PREDICTED ai_extract cost  (Azure list, $0.07/DBU):")
print(f"  Low complexity   30 DBUs/1K -> ${pred_low_list:.4f} list  /  ${pred_low_promo:.4f} with 50% promo")
print(f"  Medium           60 DBUs/1K -> ${pred_medium_list:.4f} list  /  ${pred_medium_promo:.4f} with 50% promo")
print()
print(f"Per row at low list:    ${pred_low_list / n_rows:.6f}")
print(f"Per row at medium list: ${pred_medium_list / n_rows:.6f}")

PRED = {
    "n_rows": n_rows, "total_chars": int(total_chars), "avg_chars": avg_chars,
    "pred_low_list": pred_low_list, "pred_low_promo": pred_low_promo,
    "pred_medium_list": pred_medium_list, "pred_medium_promo": pred_medium_promo,
}


## Run `ai_extract` and capture the run window

The next cell wraps the `ai_extract` call with `current_timestamp()` before and after  -  that pair of timestamps is the **reconciliation window** we'll filter `system.billing.usage` against in notebook 04.

`ai_extract` accepts a list of field names and returns a `STRUCT` with those fields populated from the input text. We ask for four fields that match what a fraud-detection tool would want:
- `caller_name`
- `account_id`
- `dollar_amount`
- `fraud_signal_phrase`

See [`ai_extract` SQL function](https://docs.databricks.com/en/sql/language-manual/functions/ai_extract.html) for the full signature.


In [ ]:
from pyspark.sql import functions as F

run_start = utc_now()
print(f"Run start (UTC): {run_start.isoformat()}")

extract_sql = f"""
    SELECT
        transcript_id,
        caller_name_truth,
        account_id_truth,
        dollar_amount_truth,
        is_fraud_truth,
        fraud_phrase_truth,
        ai_extract(
            transcript,
            ARRAY('caller_name', 'account_id', 'dollar_amount', 'fraud_signal_phrase')
        ) AS extracted,
        current_timestamp() AS run_ts
    FROM {TRANSCRIPTS_TABLE}
"""

result_df = (
    spark.sql(extract_sql)
        .withColumn("run_id", F.lit(RUN_ID))
        .withColumn("team",   F.lit(TEAM_TAG))
        .withColumn("path",   F.lit("ai_extract"))
)

(result_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(RESULTS_TABLE))

run_end = utc_now()
duration_sec = (run_end - run_start).total_seconds()
print(f"Run end   (UTC): {run_end.isoformat()}")
print(f"Duration:        {duration_sec:.1f} sec  ({duration_sec/60:.2f} min)")

display(spark.table(RESULTS_TABLE).select("transcript_id", "extracted", "fraud_phrase_truth").limit(5))


## Log the run for reconciliation

The next cell appends a row to `run_log` capturing everything notebook 04 will need:
- `run_id`, `path = 'ai_extract'`, `team`, `owner`
- `run_start`, `run_end`  -  the reconciliation window
- `n_rows`, `total_chars`, `avg_chars`  -  workload size
- `predicted_*`  -  all four prediction scenarios from the cell above

Notebook 04 reads the most recent rows from `run_log` (one per path) and joins them to `system.billing.usage`.


In [ ]:
import datetime as dt
from pyspark.sql.types import (StructType, StructField, StringType, TimestampType,
                                DoubleType, LongType)

log_schema = StructType([
    StructField("run_id",                    StringType(),    False),
    StructField("path",                      StringType(),    False),
    StructField("team",                      StringType(),    True),
    StructField("owner",                     StringType(),    True),
    StructField("run_start",                 TimestampType(), True),
    StructField("run_end",                   TimestampType(), True),
    StructField("duration_sec",              DoubleType(),    True),
    StructField("n_rows",                    LongType(),      True),
    StructField("total_chars",               LongType(),      True),
    StructField("avg_chars",                 DoubleType(),    True),
    StructField("predicted_low_list",        DoubleType(),    True),
    StructField("predicted_low_promo",       DoubleType(),    True),
    StructField("predicted_medium_list",     DoubleType(),    True),
    StructField("predicted_medium_promo",    DoubleType(),    True),
    StructField("predicted_token_reference", DoubleType(),    True),
    StructField("logged_ts",                 TimestampType(), True),
])

log_row = (
    RUN_ID, "ai_extract", TEAM_TAG, OWNER_EMAIL,
    run_start, run_end,
    float((run_end - run_start).total_seconds()),
    int(PRED["n_rows"]), int(PRED["total_chars"]), float(PRED["avg_chars"]),
    float(PRED["pred_low_list"]),    float(PRED["pred_low_promo"]),
    float(PRED["pred_medium_list"]), float(PRED["pred_medium_promo"]),
    None,                                # predicted_token_reference: 03b's responsibility
    utc_now(),
)

(spark.createDataFrame([log_row], schema=log_schema)
     .write
     .mode("append")
     .option("mergeSchema", "true")
     .saveAsTable(RUN_LOG_TABLE))

print(f"Logged run {RUN_ID} to {RUN_LOG_TABLE}")
display(spark.table(RUN_LOG_TABLE).orderBy(F.desc("logged_ts")).limit(5))


## Done

`ai_extract` results are in `extract_results`. Predicted cost + run window are in `run_log`. Notebook 04 will reconcile both against `system.billing.usage`.

**Next:** `03b_run_ai_query.ipynb` (running in parallel) does the same on `ai_query`. After both finish, `04_reconcile_costs.ipynb` does the side-by-side cost comparison.
